In [ ]:
# IMPLEMENTASI BAGGING DAN RANDOM FOREST

import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

# Bagging dengan Decision Tree
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',  # sqrt(p) untuk klasifikasi
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Perbandingan
print("=" * 50)
print("Perbandingan Akurasi:")
print(f"Decision Tree   : {accuracy_score(y_test, dt_pred):.4f}")
print(f"Bagging         : {accuracy_score(y_test, bagging_pred):.4f}")
print(f"Random Forest   : {accuracy_score(y_test, rf_pred):.4f}")

# Feature Importance dari Random Forest
importance = pd.DataFrame({
    'feature': data.feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 Feature Importance:")
print(importance.head())


In [ ]:
# IMPLEMENTASI ADABOOST DAN GRADIENT BOOSTING


from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

# AdaBoost (base estimator = decision stump)
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # decision stump
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)
ada.fit(X_train, y_train)
ada_pred = ada.predict(X_test)

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

# Feature Importance dari Gradient Boosting
gb_importance = pd.DataFrame({
    'feature': data.feature_names,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=False)

# Evaluasi
print("=" * 50)
print("Perbandingan Boosting Methods:")
print(f"AdaBoost           : {accuracy_score(y_test, ada_pred):.4f}")
print(f"Gradient Boosting  : {accuracy_score(y_test, gb_pred):.4f}")

# Learning Curve
train_scores = []
test_scores = []
estimators = [1, 10, 25, 50, 75, 100, 150, 200]

for n in estimators:
    gb_temp = GradientBoostingClassifier(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    gb_temp.fit(X_train, y_train)
    train_scores.append(gb_temp.score(X_train, y_train))
    test_scores.append(gb_temp.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(estimators, train_scores, label='Train', marker='o')
plt.plot(estimators, test_scores, label='Test', marker='s')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting Learning Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#IMPLEMENTASI STACKING

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Definisikan base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42))
]

# Meta-learner (biasanya simple model)
meta_learner = LogisticRegression(max_iter=1000)

# Stacking Classifier
stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,  # cross-validation untuk generate meta-features
    stack_method='predict_proba'  # untuk klasifikasi
)

stacking.fit(X_train, y_train)
stacking_pred = stacking.predict(X_test)

# Bandingkan semua metode
print("=" * 50)
print("PERBANDINGAN SEMUA METODE")
print("=" * 50)
print(f"Single Decision Tree : {accuracy_score(y_test, dt_pred):.4f}")
print(f"Bagging              : {accuracy_score(y_test, bagging_pred):.4f}")
print(f"Random Forest        : {accuracy_score(y_test, rf_pred):.4f}")
print(f"AdaBoost             : {accuracy_score(y_test, ada_pred):.4f}")
print(f"Gradient Boosting    : {accuracy_score(y_test, gb_pred):.4f}")
print(f"Stacking             : {accuracy_score(y_test, stacking_pred):.4f}")

# Visualisasi perbandingan
models = ['DT', 'Bagging', 'RF', 'AdaBoost', 'GB', 'Stacking']
scores = [
    accuracy_score(y_test, dt_pred),
    accuracy_score(y_test, bagging_pred),
    accuracy_score(y_test, rf_pred),
    accuracy_score(y_test, ada_pred),
    accuracy_score(y_test, gb_pred),
    accuracy_score(y_test, stacking_pred)
]

plt.figure(figsize=(10, 5))
plt.bar(models, scores, color=['blue', 'green', 'red', 'orange', 'purple', 'brown'])
plt.ylabel('Accuracy')
plt.title('Perbandingan Akurasi: Single Model vs Ensemble')
plt.ylim(0.9, 1.0)

for i, v in enumerate(scores):
    plt.text(i, v + 0.002, f'{v:.4f}', ha='center')

plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
import pandas as pd
import time

# Load dataset contoh
data = load_iris()
X = data.data
y = data.target

# Definisi stacking
estimators = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

# List model
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Stacking': stacking
}

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, model in models.items():
    start_time = time.time()

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='accuracy'
    )

    elapsed_time = time.time() - start_time

    results.append({
        'Model': name,
        'Mean Accuracy': scores.mean(),
        'Std': scores.std(),
        'Training Time (s)': elapsed_time
    })

results_df = pd.DataFrame(results).round(4)
print(results_df.to_string(index=False))

# Kesimpulan
best_model = results_df.loc[
    results_df['Mean Accuracy'].idxmax(),
    'Model'
]

print(f"\nModel terbaik berdasarkan CV: {best_model}")